In [0]:
%run ../source_to_bronze/utils

In [0]:
from pyspark.sql.functions import *

# Read Silver
silver_path = "/Volumes/workspace/assignments/dev_catalog/silver"

employee_df = spark.read.format("delta").load(f"{silver_path}/dim_employee")
department_df = spark.read.format("delta").load(f"{silver_path}/dim_department")
country_df = spark.read.format("delta").load(f"{silver_path}/dim_country")

# Join
df = employee_df \
    .join(department_df, employee_df.department == department_df.department_id, "left") \
    .join(country_df, employee_df.country == country_df.country_code, "left")



In [0]:
# =========================
# BUSINESS LOGIC
# =========================

salary_df = df.groupBy("department_name") \
    .sum("salary") \
    .withColumnRenamed("sum(salary)", "total_salary") \
    .orderBy("total_salary", ascending=False)

emp_count_df = df.groupBy("department_name", "country_name").count()

dept_country_df = df.select("department_name", "country_name").distinct()

avg_age_df = df.groupBy("department_name") \
    .agg(avg("age").alias("avg_age"))

# Add load date
salary_df = salary_df.withColumn("at_load_date", current_date())
emp_count_df = emp_count_df.withColumn("at_load_date", current_date())
dept_country_df = dept_country_df.withColumn("at_load_date", current_date())
avg_age_df = avg_age_df.withColumn("at_load_date", current_date())


In [0]:
# DISPLAY OUTPUTS (IMPORTANT)

print("Salary per Department")
display(salary_df)

print("Employee Count per Dept & Country")
display(emp_count_df)

print("Department & Country Mapping")
display(dept_country_df)

print("Average Age per Department")
display(avg_age_df)

In [0]:
# WRITE GOLD

gold_path = "/Volumes/workspace/assignments/dev_catalog/gold/employee"

salary_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_path}/fact_salary")

emp_count_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_path}/fact_emp_count")

dept_country_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_path}/dim_dept_country")

avg_age_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_path}/fact_avg_age")

print("Gold Layer Created Successfully")

In [0]:
fact_df = df.groupBy("department_name", "country_name") \
    .agg(
        sum("salary").alias("total_salary"),
        count("employee_id").alias("employee_count"),
        avg("age").alias("avg_age")
    ) \
    .withColumn("at_load_date", current_date())

fact_df.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "at_load_date = current_date()") \
    .save("/Volumes/workspace/assignments/dev_catalog/gold/employee/fact_employee")

In [0]:
display(fact_df)